In [1]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys 

sys.path.append("../src/")

In [3]:
from domain import (
    User, 
    Financial_Profile,
    Budget,
    Category,
    Budget_Category,
    Payment_Method,
    Transaction_Type,
    Transaction,
    Tag
)


In [4]:
from datetime import datetime, date
from decimal import Decimal
from typing import Optional, List
from sqlalchemy import create_engine, Integer, String, Date, Boolean, ForeignKey, Numeric, Text, DateTime, func, Table, Column, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, sessionmaker, Session

In [5]:
engine = create_engine("sqlite:///walletly.db")

class Base(DeclarativeBase):
    ...


transaction_tag = Table(
    "transaction_tag",
    Base.metadata,
    Column("id_transaction", Integer, ForeignKey("transactions.id_transaction"), primary_key=True),
    Column("id_tag", Integer, ForeignKey("tag.id_tag"), primary_key=True)

)


class User(Base):
    __tablename__ = "users"

    id_user: Mapped[int] = mapped_column(Integer, primary_key = True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(50), nullable=False)
    surname: Mapped[str] = mapped_column(String(50), nullable=False)
    email: Mapped[str] = mapped_column(String(100), nullable=False, unique = True)
    password: Mapped[str] = mapped_column(String(255), nullable=False)
    registration_date: Mapped[datetime] = mapped_column(DateTime, server_default=func.now())

    financial_profile: Mapped[Optional["Financial_Profile"]] = relationship(back_populates="user") 
    budget: Mapped[list["Budget"]] = relationship(back_populates="user")


class Financial_Profile(Base):
    __tablename__ = "financial_profile"

    id_profile: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    id_user: Mapped[int] = mapped_column(ForeignKey("users.id_user"), unique=True, nullable=False)
    monthly_salary: Mapped[Decimal] = mapped_column(Numeric(10,2), default=0.00)
    savings_goal: Mapped[Decimal] = mapped_column(Numeric(10,2), default=0.00)
    currency: Mapped[str] = mapped_column(String(3), server_default = "eur")
    risk_profile: Mapped[str] = mapped_column(String(20), nullable=True)

    user: Mapped["User"] = relationship(back_populates="financial_profile")



class Budget(Base):
    __tablename__ = "budget"

    id_budget: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    id_user: Mapped[int] = mapped_column(ForeignKey("users.id_user"), nullable=False)
    month: Mapped[int] = mapped_column(Integer, nullable=False)
    year: Mapped[int] = mapped_column(Integer, nullable=False)
    description: Mapped[str] = mapped_column(Text, nullable=True)
    total_limit: Mapped[Decimal] = mapped_column(Numeric(10,2), nullable=True)

    transactions: Mapped[list["Transaction"]] = relationship(back_populates="budget")
    user: Mapped["User"] = relationship(back_populates="budget")
    budget_categories: Mapped[list["Budget_Category"]] = relationship(back_populates="budget")
    
    

class Category(Base):
    __tablename__ = "category"

    id_category: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(50), nullable=False)
    description: Mapped[str] = mapped_column(String(255), nullable=True)
    is_essential: Mapped[bool] = mapped_column(Boolean, server_default="false", default=False)

    transactions: Mapped[list["Transaction"]] = relationship(back_populates="category")
    budget_categories: Mapped[list["Budget_Category"]] = relationship(back_populates="category")
    


class Budget_Category(Base):
    __tablename__ = "budget_category"

    id_budget: Mapped[int] = mapped_column(ForeignKey("budget.id_budget"), primary_key=True)
    id_category: Mapped[int] = mapped_column(ForeignKey("category.id_category"), primary_key=True)
    max_amount: Mapped[Decimal] = mapped_column(Numeric(10, 2), nullable=False)
    alert_threshold: Mapped[Decimal] = mapped_column(Numeric(5, 2), server_default="0.80", default=0.80)

    budget: Mapped["Budget"] = relationship(back_populates="budget_categories")
    category: Mapped["Category"] = relationship(back_populates="budget_categories")


class Payment_Method(Base):
    __tablename__ = "payment_method"

    id_method: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(50), nullable=False)
    provider: Mapped[str] = mapped_column(String(50), nullable=True)

    transactions: Mapped[list["Transaction"]] = relationship(back_populates="payment_method")



class Transaction_Type(Base):
    __tablename__ = "transaction_type"

    id_type: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(50), nullable=False)

    transactions: Mapped[list["Transaction"]] = relationship(back_populates="transaction_type")



class Transaction(Base):
    __tablename__ = "transactions"

    id_transaction: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    id_budget: Mapped[int] = mapped_column(ForeignKey("budget.id_budget"), nullable=True)
    id_method: Mapped[int] = mapped_column(ForeignKey("payment_method.id_method"), nullable=True)
    id_type: Mapped[int] = mapped_column(ForeignKey("transaction_type.id_type"), nullable=True)
    id_category: Mapped[int] = mapped_column(ForeignKey("category.id_category"), nullable=True)

    amount: Mapped[Decimal] = mapped_column(Numeric(10, 2), nullable=False)
    date: Mapped[date] = mapped_column(Date, nullable=False)
    description: Mapped[str] = mapped_column(String(255), nullable=True)
    is_recurring: Mapped[bool] = mapped_column(Boolean, server_default="false", default=False)

    category: Mapped["Category"] = relationship(back_populates="transactions")
    budget: Mapped["Budget"] = relationship(back_populates="transactions")
    payment_method: Mapped["Payment_Method"] = relationship(back_populates="transactions")
    transaction_type: Mapped["Transaction_Type"] = relationship(back_populates="transactions")

    tags: Mapped[list["Tag"]] = relationship(
        secondary = transaction_tag,
        back_populates="transactions"
    )


class Tag(Base):
    __tablename__ = "tag"
    
    id_tag: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(50), nullable=False, unique=True)
    color_code: Mapped[str] = mapped_column(String(7), nullable=True)

    transactions: Mapped[list["Transaction"]] = relationship(
        secondary="transaction_tag",
        back_populates="tags"
    )
    

Base.metadata.create_all(engine)


In [6]:
with Session(engine) as session:
    type_names = ["Income", "Expense", "Transfer"]

    existing_types = {t.name: t for t in session.query(Transaction_Type).all()}

for name in type_names:
    if name not in existing_types:
        new_type = Transaction_Type(name=name)
        session.add(new_type)
        session.flush()
        existing_types[name] = new_type

session.commit()
print("Transaction types initialized successfully.")

Transaction types initialized successfully.
